# Milestone 4: Unified Training, Validation, and Hyperparameter Tuning

This notebook is now focused on **Milestone 3 and Milestone 4**, not a deep Milestone 2-style dataset audit.

## What this notebook covers

1. Prepare the final 4-class training dataset
2. Build one unified experiment grid
3. Treat `YOLOv8n`, `YOLOv8s`, `YOLOv8m`, and `YOLOv8l` as model-size hyperparameters
4. Train multiple configurations and validate each one
5. Compare all runs in one summary table
6. Select the final winning configuration
7. Inspect predictions from the best model
8. Optionally export and benchmark for deployment

## Submission note

This is the **main submission notebook**. The code is written so you can run it top-to-bottom. Long training cells are protected by boolean flags so nothing expensive starts by accident.


In [1]:
# Install once if needed.
%pip install -q ultralytics roboflow opencv-python-headless pandas numpy matplotlib seaborn pillow pyyaml tqdm scikit-learn iterative-stratification


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 41.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 118.6 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import platform
import random
import shutil
import time
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from IPython.display import Markdown, display
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from PIL import Image
from ultralytics import YOLO

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

SEED = 42
TARGET_IMG_SIZE = 640
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.80, 0.10, 0.10

PROJECT_ROOT = Path.cwd()
LOCAL_DATASET_ROOT = PROJECT_ROOT / "RDD2022-India-5"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "milestone4"
DATASET_OUT = ARTIFACT_ROOT / "datasets" / "final4_stratified_801010"
TRAINING_ROOT = ARTIFACT_ROOT / "training_runs"
REPORT_ROOT = ARTIFACT_ROOT / "reports"
EXPORT_ROOT = ARTIFACT_ROOT / "exports"

for directory in [ARTIFACT_ROOT, DATASET_OUT.parent, TRAINING_ROOT, REPORT_ROOT, EXPORT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

RAW_CLASS_NAMES = {
    0: "D00",
    1: "D01",
    2: "D0w0",
    3: "D10",
    4: "D11",
    5: "D20",
    6: "D40",
    7: "D43",
    8: "D44",
    9: "D50",
}
FINAL_CLASS_MAP = {
    0: 0,
    1: 0,
    2: 0,
    3: 1,
    4: 1,
    5: 2,
    6: 3,
}
FINAL_CLASS_NAMES = {
    0: "Longitudinal Crack",
    1: "Transverse Crack",
    2: "Alligator Crack",
    3: "Pothole",
}

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

training_device = 0 if torch.cuda.is_available() else "cpu"
device_label = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"

display(
    Markdown(
        f'''
### Notebook Configuration

- `PROJECT_ROOT`: `{PROJECT_ROOT}`
- `LOCAL_DATASET_ROOT`: `{LOCAL_DATASET_ROOT}`
- `DATASET_OUT`: `{DATASET_OUT}`
- `TARGET_IMG_SIZE`: `{TARGET_IMG_SIZE}`
- `SEED`: `{SEED}`
- `TRAINING DEVICE`: `{device_label}`
'''
    )
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.



### Notebook Configuration

- `PROJECT_ROOT`: `/kaggle/working`
- `LOCAL_DATASET_ROOT`: `/kaggle/working/RDD2022-India-5`
- `DATASET_OUT`: `/kaggle/working/artifacts/milestone4/datasets/final4_stratified_801010`
- `TARGET_IMG_SIZE`: `640`
- `SEED`: `42`
- `TRAINING DEVICE`: `Tesla T4`


## 1. Dataset Setup for Training

This section keeps only what is necessary for Milestone 3 and 4:

- load the local dataset or download it if missing
- merge raw classes into the final 4-class task
- create a stratified `80/10/10` split
- save the final `data.yaml`

Everything else should focus on training, validation, model comparison, and tuning.


In [3]:
if LOCAL_DATASET_ROOT.exists():
    CURRENT_DATASET_ROOT = LOCAL_DATASET_ROOT.resolve()
    print(f"Using existing local dataset: {CURRENT_DATASET_ROOT}")
else:
    from roboflow import Roboflow

    # roboflow_api_key = os.getenv("ROBOFLOW_API_KEY")
    # if not roboflow_api_key:
    #     raise EnvironmentError(
    #         "Local dataset folder is missing and ROBOFLOW_API_KEY is not set. "
    #         "Either keep RDD2022-India-5 in the workspace or export ROBOFLOW_API_KEY."
    #     )

    rf = Roboflow(api_key="zCX5IJCFMWJsI0xzXTWE")
    project = rf.workspace("prakhar-kpb1v").project("rdd2022-india-il8ju")
    version = project.version(5)
    dataset = version.download("yolov8")
    CURRENT_DATASET_ROOT = Path(dataset.location).resolve()
    print(f"Dataset downloaded to: {CURRENT_DATASET_ROOT}")

CURRENT_DATASET_ROOT


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to RDD2022-India-5 in yolov8:: 100%|██████████| 15424/15424 [00:01<00:00, 8915.79it/s] 


Dataset downloaded to: /kaggle/working/RDD2022-India-5


PosixPath('/kaggle/working/RDD2022-India-5')

In [4]:
def available_split_dirs(dataset_root: Path) -> dict[str, Path]:
    candidates = {}
    for canonical, aliases in {
        "train": ["train"],
        "val": ["val", "valid", "validation"],
        "test": ["test"],
    }.items():
        for alias in aliases:
            candidate = dataset_root / alias
            if candidate.exists():
                candidates[canonical] = candidate
                break
    return candidates

def find_image_for_label(label_path: Path, image_dir: Path) -> Path | None:
    exact = image_dir / f"{label_path.stem}.jpg"
    if exact.exists():
        return exact
    matches = list(image_dir.glob(f"{label_path.stem}.*"))
    return matches[0] if matches else None

def parse_yolo_label_file(label_path: Path, width: int, height: int, class_names: dict[int, str]) -> list[dict]:
    rows = []
    with label_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            class_id = int(float(parts[0]))
            x_center, y_center, box_w, box_h = map(float, parts[1:])
            rows.append(
                {
                    "class_id": class_id,
                    "class_name": class_names.get(class_id, str(class_id)),
                    "x_center": x_center,
                    "y_center": y_center,
                    "width": box_w,
                    "height": box_h,
                    "bbox_area_px": (box_w * width) * (box_h * height),
                }
            )
    return rows

def collect_dataset_frames(dataset_root: Path, class_names: dict[int, str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    image_rows = []
    box_rows = []
    for split_name, split_dir in available_split_dirs(dataset_root).items():
        image_dir = split_dir / "images"
        label_dir = split_dir / "labels"
        for label_path in sorted(label_dir.glob("*.txt")):
            image_path = find_image_for_label(label_path, image_dir)
            if image_path is None:
                continue
            with Image.open(image_path) as image:
                width, height = image.size
            labels = parse_yolo_label_file(label_path, width, height, class_names=class_names)
            image_rows.append(
                {
                    "split": split_name,
                    "image_path": str(image_path),
                    "label_path": str(label_path),
                    "image_name": image_path.name,
                    "width": width,
                    "height": height,
                    "class_ids": sorted({int(label["class_id"]) for label in labels}),
                    "n_objects": len(labels),
                }
            )
            for label in labels:
                label["split"] = split_name
                label["image_path"] = str(image_path)
                box_rows.append(label)
    return pd.DataFrame(image_rows), pd.DataFrame(box_rows)

def save_yaml(data: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        yaml.safe_dump(data, handle, sort_keys=False)

images_df, boxes_df = collect_dataset_frames(CURRENT_DATASET_ROOT, RAW_CLASS_NAMES)
print(f"Images found: {len(images_df):,}")
print(f"Boxes found: {len(boxes_df):,}")


Images found: 7,706
Boxes found: 8,203


In [5]:
all_images = images_df.copy().reset_index(drop=True)
multilabel_targets = np.zeros((len(all_images), len(FINAL_CLASS_NAMES) + 1), dtype=int)

final_class_presence = []
for class_ids in all_images["class_ids"]:
    mapped = sorted({FINAL_CLASS_MAP[c] for c in class_ids if c in FINAL_CLASS_MAP})
    final_class_presence.append(mapped)

all_images["final_class_ids"] = final_class_presence
for idx, mapped_ids in enumerate(final_class_presence):
    if mapped_ids:
        for class_id in mapped_ids:
            multilabel_targets[idx, class_id] = 1
    else:
        multilabel_targets[idx, -1] = 1  # background / no in-scope defect

first_split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, temp_idx = next(first_split.split(np.zeros(len(all_images)), multilabel_targets))

second_split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_rel_idx, test_rel_idx = next(
    second_split.split(np.zeros(len(temp_idx)), multilabel_targets[temp_idx])
)

val_idx = temp_idx[val_rel_idx]
test_idx = temp_idx[test_rel_idx]

split_assignment = pd.Series(index=all_images.index, dtype="object")
split_assignment.iloc[train_idx] = "train"
split_assignment.iloc[val_idx] = "val"
split_assignment.iloc[test_idx] = "test"
all_images["final_split"] = split_assignment.values

if DATASET_OUT.exists():
    shutil.rmtree(DATASET_OUT)
for split_name in ["train", "val", "test"]:
    (DATASET_OUT / split_name / "images").mkdir(parents=True, exist_ok=True)
    (DATASET_OUT / split_name / "labels").mkdir(parents=True, exist_ok=True)

retained_counter = Counter()
for _, row in all_images.iterrows():
    src_image = Path(row["image_path"])
    src_label = Path(row["label_path"])
    split_name = row["final_split"]
    new_lines = []
    with src_label.open("r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            raw_class_id = int(float(parts[0]))
            if raw_class_id in FINAL_CLASS_MAP:
                final_class_id = FINAL_CLASS_MAP[raw_class_id]
                retained_counter[FINAL_CLASS_NAMES[final_class_id]] += 1
                new_lines.append(" ".join([str(final_class_id)] + parts[1:]))

    shutil.copy2(src_image, DATASET_OUT / split_name / "images" / src_image.name)
    target_label = DATASET_OUT / split_name / "labels" / src_label.name
    with target_label.open("w", encoding="utf-8") as handle:
        if new_lines:
            handle.write("\n".join(new_lines) + "\n")

final_yaml = {
    "path": str(DATASET_OUT.resolve()),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": [FINAL_CLASS_NAMES[i] for i in range(len(FINAL_CLASS_NAMES))],
    "nc": len(FINAL_CLASS_NAMES),
}
save_yaml(final_yaml, DATASET_OUT / "data.yaml")

final_images_df, final_boxes_df = collect_dataset_frames(DATASET_OUT, FINAL_CLASS_NAMES)
split_summary = final_images_df.groupby("split").size().rename("images").reset_index()
split_summary["fraction"] = split_summary["images"] / split_summary["images"].sum()
class_summary = (
    final_boxes_df.groupby(["class_id", "class_name"]).size().rename("boxes").reset_index().sort_values("class_id")
)
class_summary["percentage"] = class_summary["boxes"] / class_summary["boxes"].sum()

display(split_summary)
display(class_summary)

display(
    Markdown(
        f'''
### Final Training Dataset Ready

- Output dataset: `{DATASET_OUT}`
- Final split: **{split_summary.loc[split_summary["split"] == "train", "fraction"].iloc[0]:.2%} / {split_summary.loc[split_summary["split"] == "val", "fraction"].iloc[0]:.2%} / {split_summary.loc[split_summary["split"] == "test", "fraction"].iloc[0]:.2%}**
- Final classes: **Longitudinal Crack, Transverse Crack, Alligator Crack, Pothole**
- Input resolution in the source dataset remains **720 x 720** and training will use **{TARGET_IMG_SIZE} x {TARGET_IMG_SIZE}**

This is the preprocessing part that Milestone 3 and 4 actually need before training starts.
'''
    )
)


,split,images,fraction
0,test,765,0.099273
1,train,6168,0.800415
2,val,773,0.100311


,class_id,class_name,boxes,percentage
0,0,Longitudinal Crack,1735,0.245890
1,1,Transverse Crack,113,0.016015
2,2,Alligator Crack,2021,0.286423
3,3,Pothole,3187,0.451672



### Final Training Dataset Ready

- Output dataset: `/kaggle/working/artifacts/milestone4/datasets/final4_stratified_801010`
- Final split: **80.04% / 10.03% / 9.93%**
- Final classes: **Longitudinal Crack, Transverse Crack, Alligator Crack, Pothole**
- Input resolution in the source dataset remains **720 x 720** and training will use **640 x 640**

This is the preprocessing part that Milestone 3 and 4 actually need before training starts.


## 2. Unified Hyperparameter Grid

This notebook now uses **one single experiment grid**.

In this grid, the following are all treated as hyperparameters:

- model variant: `YOLOv8n`, `YOLOv8s`, `YOLOv8m`, `YOLOv8l`
- image size
- batch size
- learning rate
- augmentation profile
- pretrained vs scratch

So model choice is no longer separated from tuning. It is part of the same unified search.


In [6]:
hardware_df = pd.DataFrame(
    [
        {"setting": "platform", "value": platform.platform()},
        {"setting": "python", "value": platform.python_version()},
        {"setting": "torch", "value": torch.__version__},
        {"setting": "device", "value": device_label},
        {"setting": "cpu_cores", "value": os.cpu_count()},
    ]
)
display(hardware_df)

TRAINING_COMMON_ARGS = {
    "data": str((DATASET_OUT / "data.yaml").resolve()),
    "epochs": 30,
    "imgsz": TARGET_IMG_SIZE,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "patience": 20,
    "seed": SEED,
    "deterministic": True,
    "project": str(TRAINING_ROOT.resolve()),
    "exist_ok": True,
    "device": training_device,
    "workers": 4,
    "save": True,
    "save_period": 10,
    "plots": True,
    "pretrained": True,
    "hsv_h": 0.015,
    "hsv_s": 0.50,
    "hsv_v": 0.40,
    "degrees": 7.5,
    "translate": 0.10,
    "scale": 0.30,
    "fliplr": 0.50,
    "flipud": 0.0,
    "mosaic": 0.60,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "close_mosaic": 10,
}

AUGMENTATION_PROFILES = {
    "standard": {},
    "none": {
        "hsv_h": 0.0,
        "hsv_s": 0.0,
        "hsv_v": 0.0,
        "degrees": 0.0,
        "translate": 0.0,
        "scale": 0.0,
        "fliplr": 0.0,
        "mosaic": 0.0,
    },
}

BATCH_BY_VARIANT = {
    "yolov8n.pt": 24,
    "yolov8s.pt": 16,
    "yolov8m.pt": 12,
    "yolov8l.pt": 8,
}

display(pd.DataFrame({"argument": list(TRAINING_COMMON_ARGS.keys()), "value": list(TRAINING_COMMON_ARGS.values())}))


,setting,value
0,platform,Linux-6.6.113+-x86_64-with-glibc2.35
1,python,3.12.12
2,torch,2.10.0+cu128
3,device,Tesla T4
4,cpu_cores,4


,argument,value
0,data,/kaggle/working/artifacts/milestone4/datasets/...
1,epochs,30
2,imgsz,640
3,optimizer,AdamW
4,lr0,0.001
5,lrf,0.01
6,momentum,0.937
7,weight_decay,0.0005
8,warmup_epochs,3.0
9,box,7.5


In [7]:
EXPERIMENT_GRID = [

    {
        "experiment_name": "grid_yolov8s_896_lr1e3_aug",
        "weights": "yolov8s.pt",
        "batch": 12,
        "imgsz": 896,
        "lr0": 0.001,
        "augmentation_profile": "standard",
        "pretrained": True,
        "purpose": "Higher resolution for small model",
    },
    {
        "experiment_name": "grid_yolov8m_896_lr1e3_aug",
        "weights": "yolov8m.pt",
        "batch": 8,
        "imgsz": 896,
        "lr0": 0.001,
        "augmentation_profile": "standard",
        "pretrained": True,
        "purpose": "Higher resolution for medium model",
    },
    {
        "experiment_name": "grid_yolov8m_640_lr5e4_aug",
        "weights": "yolov8m.pt",
        "batch": BATCH_BY_VARIANT["yolov8m.pt"],
        "imgsz": 640,
        "lr0": 0.0005,
        "augmentation_profile": "standard",
        "pretrained": True,
        "purpose": "Lower learning rate for medium model",
    },
    {
        "experiment_name": "grid_yolov8m_640_lr2e3_aug",
        "weights": "yolov8m.pt",
        "batch": BATCH_BY_VARIANT["yolov8m.pt"],
        "imgsz": 640,
        "lr0": 0.0020,
        "augmentation_profile": "standard",
        "pretrained": True,
        "purpose": "Higher learning rate for medium model",
    },
    {
        "experiment_name": "grid_yolov8m_640_lr1e3_noaug",
        "weights": "yolov8m.pt",
        "batch": BATCH_BY_VARIANT["yolov8m.pt"],
        "imgsz": 640,
        "lr0": 0.001,
        "augmentation_profile": "none",
        "pretrained": True,
        "purpose": "Quantify augmentation effect",
    },
    {
        "experiment_name": "grid_yolov8m_640_lr1e3_scratch",
        "weights": "yolov8m.yaml",
        "batch": BATCH_BY_VARIANT["yolov8m.pt"],
        "imgsz": 640,
        "lr0": 0.001,
        "augmentation_profile": "standard",
        "pretrained": False,
        "purpose": "Quantify transfer-learning effect",
    },
]

experiment_grid_df = pd.DataFrame(EXPERIMENT_GRID)
experiment_grid_df["model_variant"] = experiment_grid_df["weights"].str.replace(".pt", "", regex=False).str.replace(".yaml", "", regex=False)
display(experiment_grid_df)


,experiment_name,weights,batch,imgsz,lr0,augmentation_profile,pretrained,purpose,model_variant
0,grid_yolov8s_896_lr1e3_aug,yolov8s.pt,12,896,0.0010,standard,True,Higher resolution for small model,yolov8s
1,grid_yolov8m_896_lr1e3_aug,yolov8m.pt,8,896,0.0010,standard,True,Higher resolution for medium model,yolov8m
2,grid_yolov8m_640_lr5e4_aug,yolov8m.pt,12,640,0.0005,standard,True,Lower learning rate for medium model,yolov8m
3,grid_yolov8m_640_lr2e3_aug,yolov8m.pt,12,640,0.0020,standard,True,Higher learning rate for medium model,yolov8m
4,grid_yolov8m_640_lr1e3_noaug,yolov8m.pt,12,640,0.0010,none,True,Quantify augmentation effect,yolov8m
5,grid_yolov8m_640_lr1e3_scratch,yolov8m.yaml,12,640,0.0010,standard,False,Quantify transfer-learning effect,yolov8m


In [8]:
def run_experiment(experiment: dict) -> dict:
    args = TRAINING_COMMON_ARGS.copy()
    args.update(
        {
            "name": experiment["experiment_name"],
            "batch": experiment["batch"],
            "imgsz": experiment["imgsz"],
            "pretrained": experiment.get("pretrained", True),
            "lr0": experiment["lr0"],
        }
    )
    args.update(AUGMENTATION_PROFILES[experiment["augmentation_profile"]])
    args.update(experiment.get("overrides", {}))

    model = YOLO(experiment["weights"])
    model.train(**args)

    run_dir = TRAINING_ROOT / experiment["experiment_name"]
    best_weights = run_dir / "weights" / "best.pt"
    best_model = YOLO(str(best_weights))

    val_metrics = best_model.val(
        data=str((DATASET_OUT / "data.yaml").resolve()),
        split="val",
        imgsz=experiment["imgsz"],
        batch=max(1, min(experiment["batch"], 8)),
        device=training_device,
        plots=False,
    )
    test_metrics = best_model.val(
        data=str((DATASET_OUT / "data.yaml").resolve()),
        split="test",
        imgsz=experiment["imgsz"],
        batch=max(1, min(experiment["batch"], 8)),
        device=training_device,
        plots=False,
    )

    return {
        "experiment_name": experiment["experiment_name"],
        "model_variant": experiment["weights"].replace(".pt", "").replace(".yaml", ""),
        "weights": experiment["weights"],
        "purpose": experiment["purpose"],
        "imgsz": experiment["imgsz"],
        "batch": experiment["batch"],
        "lr0": experiment["lr0"],
        "augmentation_profile": experiment["augmentation_profile"],
        "pretrained": experiment["pretrained"],
        "val_precision": float(val_metrics.box.mp),
        "val_recall": float(val_metrics.box.mr),
        "val_map50": float(val_metrics.box.map50),
        "val_map50_95": float(val_metrics.box.map),
        "test_precision": float(test_metrics.box.mp),
        "test_recall": float(test_metrics.box.mr),
        "test_map50": float(test_metrics.box.map50),
        "test_map50_95": float(test_metrics.box.map),
        "best_weights": str(best_weights.resolve()),
    }


In [9]:
RUN_EXPERIMENT_GRID = True
experiment_grid_summary_path = REPORT_ROOT / "experiment_grid_summary.csv"

if RUN_EXPERIMENT_GRID:
    experiment_rows = []
    for experiment in EXPERIMENT_GRID:
        experiment_rows.append(run_experiment(experiment))
    experiment_results_df = pd.DataFrame(experiment_rows).sort_values("val_map50", ascending=False)
    experiment_results_df.to_csv(experiment_grid_summary_path, index=False)
    display(experiment_results_df)
else:
    display(
        Markdown(
            '''
### Training Code: Unified Experiment Grid

This is the main training cell for the notebook.

Set `RUN_EXPERIMENT_GRID = True` and run this cell to execute the full hyperparameter grid.

The grid includes:

- model variants: `YOLOv8n`, `YOLOv8s`, `YOLOv8m`, `YOLOv8l`
- image sizes: `640`, `896`
- learning rates: `0.0005`, `0.001`, `0.002`
- augmentation profiles: `standard`, `none`
- pretrained and scratch settings

Validation is checked after every run, and test metrics are also recorded in the same summary table.
'''
        )
    )


Ultralytics 8.4.31 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/artifacts/milestone4/datasets/final4_stratified_801010/data.yaml, degrees=7.5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=896, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.6, multi_scale=0.0, name=grid_yolov8s_896_lr1e3_aug, nbs=64, nms=False, opset=None, optimiz

KeyboardInterrupt: 

In [1]:
if experiment_grid_summary_path.exists():
    experiment_results_df = pd.read_csv(experiment_grid_summary_path).sort_values("val_map50", ascending=False)
    display(experiment_results_df)

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    sns.barplot(data=experiment_results_df, x="experiment_name", y="val_map50", ax=axes[0], color="#2A9D8F")
    axes[0].set_title("Validation mAP@0.5")
    axes[0].tick_params(axis="x", rotation=70)
    sns.barplot(data=experiment_results_df, x="experiment_name", y="test_map50", ax=axes[1], color="#E76F51")
    axes[1].set_title("Test mAP@0.5")
    axes[1].tick_params(axis="x", rotation=70)
    sns.barplot(data=experiment_results_df, x="experiment_name", y="test_recall", ax=axes[2], color="#264653")
    axes[2].set_title("Test Recall")
    axes[2].tick_params(axis="x", rotation=70)
    plt.tight_layout()
    plt.show()

    pivot_map = experiment_results_df.pivot_table(
        index="model_variant",
        columns="imgsz",
        values="val_map50",
        aggfunc="max",
    )
    display(pivot_map)

    best_experiment = experiment_results_df.iloc[0]
    display(
        Markdown(
            f'''
    ### Best Configuration from the Unified Grid

    - Best experiment: **{best_experiment["experiment_name"]}**
    - Model variant: **{best_experiment["model_variant"]}**
    - Image size: **{int(best_experiment["imgsz"])}**
    - Batch size: **{int(best_experiment["batch"])}**
    - Learning rate: **{best_experiment["lr0"]:.4g}**
    - Augmentation profile: **{best_experiment["augmentation_profile"]}**
    - Pretrained: **{bool(best_experiment["pretrained"])}**
    - Validation `mAP@0.5`: **{best_experiment["val_map50"]:.4f}**
    - Test `mAP@0.5`: **{best_experiment["test_map50"]:.4f}**

    This single section now answers:

    - where training happens,
    - where validation is checked,
    - how multiple model variants are compared,
    - and which full hyperparameter configuration wins.
    '''
        )
    )
else:
    display(Markdown("### Experiment-grid results are not available yet. Run the unified training cell above first."))


NameError: name 'experiment_grid_summary_path' is not defined

In [2]:
def sample_prediction_images(df: pd.DataFrame, split: str = "test", max_images: int = 6) -> list[str]:
    subset = df.query("split == @split")
    if subset.empty:
        return []
    sample_df = subset.sample(min(max_images, len(subset)), random_state=SEED)
    return sample_df["image_path"].tolist()

final_weights = None
if experiment_grid_summary_path.exists():
    final_weights = pd.read_csv(experiment_grid_summary_path).sort_values("val_map50", ascending=False).iloc[0]["best_weights"]

if final_weights:
    predictor = YOLO(str(final_weights))
    image_paths = sample_prediction_images(final_images_df, split="test", max_images=6)
    results = predictor.predict(image_paths, imgsz=TARGET_IMG_SIZE, conf=0.25, device=training_device, verbose=False)

    fig, axes = plt.subplots(len(results), 1, figsize=(12, 5 * len(results)))
    if len(results) == 1:
        axes = [axes]
    for axis, result in zip(axes, results):
        rendered = result.plot(conf=True, labels=True, line_width=2)
        axis.imshow(cv2.cvtColor(rendered, cv2.COLOR_BGR2RGB))
        axis.set_title(Path(result.path).name)
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    display(Markdown("### No trained model is available yet for qualitative predictions."))


NameError: name 'pd' is not defined

In [ ]:
# Optional deployment benchmarking after final model selection.
# Uncomment the install below if you want ONNX / OpenVINO export.
# %pip install -q onnx onnxruntime openvino nncf

def benchmark_predictor(model_path: str | Path, image_paths: list[str], runs: int = 30) -> dict:
    predictor = YOLO(str(model_path))
    sample_paths = image_paths[: runs + 5]
    warmup_paths = sample_paths[:5]
    eval_paths = sample_paths[5:]

    for path in warmup_paths:
        predictor.predict(path, imgsz=TARGET_IMG_SIZE, device="cpu", verbose=False)

    timings = []
    for path in eval_paths:
        start = time.perf_counter()
        predictor.predict(path, imgsz=TARGET_IMG_SIZE, device="cpu", verbose=False)
        timings.append((time.perf_counter() - start) * 1000.0)

    return {
        "mean_latency_ms": float(np.mean(timings)),
        "p95_latency_ms": float(np.percentile(timings, 95)),
        "fps": float(1000.0 / np.mean(timings)),
    }

BENCHMARK_FINAL_MODEL = False

if BENCHMARK_FINAL_MODEL and final_weights:
    test_images = sample_prediction_images(final_images_df, split="test", max_images=40)
    benchmark_df = pd.DataFrame(
        [{"format": "pytorch_best", **benchmark_predictor(final_weights, test_images)}]
    )
    display(benchmark_df)
else:
    display(
        Markdown(
            '''
### Optional Benchmark Cell

Use this only after training if you want CPU latency numbers for the final selected model.
'''
        )
    )


In [ ]:
artifact_rows = []
for path in sorted(ARTIFACT_ROOT.rglob("*")):
    if path.is_file():
        artifact_rows.append(
            {
                "path": str(path.resolve()),
                "size_kb": round(path.stat().st_size / 1024, 2),
                "suffix": path.suffix,
            }
        )
artifacts_df = pd.DataFrame(artifact_rows)
display(artifacts_df.head(100))


## Final Flow

If you are running this notebook for the submission, the order is:

1. run the dataset setup cells
2. inspect the unified experiment grid
3. set `RUN_EXPERIMENT_GRID = True`
4. run the unified training cell
5. inspect validation and test results from the single summary table
6. inspect the final selected model
7. generate qualitative prediction screenshots
8. optionally benchmark the final model

This version is intentionally centered on **training**, **validation**, **multi-model comparison**, and **unified hyperparameter tuning**, because those are the Milestone 3 and 4 deliverables you asked to prioritize.
